# NonNULL Framework V1.0 — Google Colab (Chạy 1-Click Toàn trình)

**Tác giả**: Giang Nhân Nghĩa (*The Non-Null Project*)  
Notebook này cho phép bạn chạy trọn bộ **NonNULL Framework V1.0** trực tiếp trên Google Colab (khuyên dùng **L4 24GB** cho Qwen-3B/7B hoặc **A100 80GB** cho Llama-3.1-8B + HarmBench/MT-Bench):
1. Kiểm tra GPU & Cài đặt thư viện
2. Chạy bộ kiểm thử đơn vị (`unittest`)
3. Huấn luyện SFT + `EulerPhaseLoss` + `ContextPairedValueCritic`
4. Căn chỉnh DPO Thấu cảm (Vòng 1)
5. Đánh giá 25 Kịch bản Lõi + Tự động trích xuất Vết sẹo Dữ liệu (*Data Scars*)
6. Vòng lặp Tự chữa lành Vết sẹo (Vòng 2)
7. Đánh giá Ngoại suy `OOD-20` & Kiểm thử Đa lượt / Bảo toàn IQ Điểm Không ($Z$)
8. Đánh giá Benchmark Cộng đồng Công khai (**HarmBench** + **MT-Bench**)
9. Giải mã Chấm mù & Tính độ đồng thuận **Cohen's Kappa** (`score_blind_eval.py`)


## 1. Kiểm tra GPU & Chuẩn bị Mã nguồn


In [ ]:
import torch, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

# Tùy chọn A: Clone từ GitHub (thay URL repo của bạn sau khi đẩy lên GitHub)
!git clone https://github.com/GiangNhanNghia/NonNULL-Framework.git
%cd NonNULL-Framework

# Tùy chọn B: Giải nén nếu bạn tải tệp NonNULL_Framework_V1.0_VI.zip lên Colab
if os.path.exists("/content/NonNULL_Framework_V1.0_VI.zip"):
    !unzip -o -q /content/NonNULL_Framework_V1.0_VI.zip -d /content/
    %cd /content/NonNULL_Framework_V1.0_VI


In [ ]:
!pip install -q -r requirements.txt
!python3 -m unittest discover -s tests -v


## 2. Chọn Mô hình Nền (3B / 7B Qwen hoặc 8B Llama)


In [ ]:
# Chọn 1 trong các mô hình đã kiểm chứng:
# MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"                       # Mốc khả thi tối thiểu (L4 / T4)
# MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"           # Mốc tối ưu Qwen (L4)
MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"      # Mốc Liên kiến trúc (A100)
TAG = "llama8b"
print("Selected:", MODEL_ID, "| Tag:", TAG)
import os
os.environ["MODEL_ID"] = MODEL_ID
os.environ["TAG"] = TAG


## 3. Huấn luyện SFT (`EulerPhaseLoss`) & `ContextPairedValueCritic`


In [ ]:
!python3 run_training.py \
  --model_id "$MODEL_ID" \
  --data_path ./nonnull_train_100.jsonl \
  --output_dir ./output/euler_checkpoints \
  --adapter_name "adapter_B_vol4_$TAG" \
  --epochs 2 --batch_size 2 --learning_rate 2e-4 --seed 42


## 4. Căn chỉnh DPO Thấu cảm (Vòng 1)


In [ ]:
!python3 run_dpo_v1.py \
  --model_id "$MODEL_ID" \
  --init_adapter "./output/euler_checkpoints/adapter_B_vol4_$TAG" \
  --output_adapter "./output/euler_checkpoints/adapter_B_dpo_$TAG" \
  --dpo_data ./dpo_train_50.jsonl \
  --val_data ./dpo_val_10.jsonl \
  --epochs 1 --lr 2e-5 --beta 0.1 --empathic_gamma 0.05 --sft_weight 0.1


## 5. Đánh giá Vòng 1 (25 Kịch bản) & Trích xuất Vết sẹo Dữ liệu


In [ ]:
!python3 evaluate_genesis.py \
  --model_id "$MODEL_ID" \
  --adapter_b "./output/euler_checkpoints/adapter_B_vol4_$TAG" \
  --adapter_c "./output/euler_checkpoints/adapter_B_dpo_$TAG" \
  --include_vol4 --use_consensus --num_samples 3 \
  --test_data ./nonnull_test_25.jsonl \
  --output "eval_${TAG}_r1.json" \
  --blind_csv "blind_${TAG}_r1.csv" \
  --blind_key "blind_${TAG}_r1_key.json" \
  --scars_out "scars_${TAG}.jsonl"


## 6. Vòng lặp Tự chữa lành Vết sẹo (Vòng 2) & Tái kiểm định


In [ ]:
!python3 run_dpo_v1.py \
  --model_id "$MODEL_ID" \
  --init_adapter "./output/euler_checkpoints/adapter_B_dpo_$TAG" \
  --output_adapter "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --dpo_data ./dpo_train_50.jsonl \
  --val_data ./dpo_val_10.jsonl \
  --scars_file "scars_${TAG}.jsonl" \
  --epochs 1 --lr 2e-5 --beta 0.1 --empathic_gamma 0.05 --sft_weight 0.1

!python3 evaluate_genesis.py \
  --model_id "$MODEL_ID" \
  --adapter_b "./output/euler_checkpoints/adapter_B_vol4_$TAG" \
  --adapter_c "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --include_vol4 --use_consensus --num_samples 3 \
  --test_data ./nonnull_test_25.jsonl \
  --output "eval_${TAG}_healed.json" \
  --blind_csv "blind_${TAG}_healed.csv" \
  --blind_key "blind_${TAG}_healed_key.json"


## 7. Kiểm thử Ngoại suy `OOD-20` & Đa lượt / Bảo toàn IQ Điểm Không ($Z$)


In [ ]:
!python3 evaluate_genesis.py \
  --model_id "$MODEL_ID" \
  --adapter_b "./output/euler_checkpoints/adapter_B_vol4_$TAG" \
  --adapter_c "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --include_vol4 --use_consensus --num_samples 3 \
  --test_data ./nonnull_ood_20.jsonl \
  --output "eval_${TAG}_ood20.json" \
  --blind_csv "blind_${TAG}_ood20.csv" \
  --blind_key "blind_${TAG}_ood20_key.json"

!python3 evaluate_extended.py \
  --model_id "$MODEL_ID" \
  --adapter_c "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --output "eval_${TAG}_extended.json"


## 8. Benchmark Cộng đồng Công khai: HarmBench & MT-Bench


In [ ]:
# Chạy song song BASE vs NonNULL trên 100 mẫu HarmBench (kèm bộ phân loại chính thức) và 80 câu MT-Bench
!python3 eval_public_benchmarks.py \
  --bench all \
  --model_id "$MODEL_ID" \
  --adapter_path "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --harmbench_subset standard \
  --harmbench_samples 100 \
  --run_harmbench_cls \
  --mtbench_samples 80 \
  --output_dir "./benchmark_outputs_${TAG}"


## 9. Giải mã Chấm mù & Tính Độ đồng thuận Cohen's Kappa


In [ ]:
# Chạy sau khi 2 giám khảo độc lập đã điền điểm vào tệp blind_${TAG}_healed.csv
# !python3 score_blind_eval.py \
#   --csv "blind_${TAG}_healed.csv" \
#   --key "blind_${TAG}_healed_key.json" \
#   --out_md "blind_${TAG}_summary.md"
